In [15]:
import operator
from google import genai
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END

In [16]:
load_dotenv()

client = genai.Client()

In [17]:
class UPSCState(TypedDict):

    essay_topic: str
    essay_generated: str

    clarity_of_thought_feedback: str
    depth_of_analysis_feedback: str
    language_used_feedback: str

    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: int

In [18]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description= "Textual Feedback of the Essay")
    score: int = Field(description= "Score of the Essay out of 10", ge=0, le=10)

In [19]:
def generate_essay(State: UPSCState) -> UPSCState:

    essay_topic = State['essay_topic']

    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = f"Generate an essay on the topic: {essay_topic}, make some mistakes and grammatical errors like a human"
    )

    return {'essay_generated': response.text}

In [20]:
def clarity_of_thought_evaluation(State: UPSCState) -> UPSCState:

    essay_generated = State['essay_generated']

    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = f"Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10\n{essay_generated}",
        config = {
            'response_mime_type': 'application/json',
            'response_schema': EvaluationSchema
        }
    )

    return {
        'clarity_of_thought_feedback': response.parsed.feedback,
        'individual_scores': [response.parsed.score]
    }

In [21]:
def depth_of_analysis_evaluation(State: UPSCState) -> UPSCState:

    essay_generated = State['essay_generated']

    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = f"Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10\n{essay_generated}",
        config = {
            'response_mime_type': 'application/json',
            'response_schema': EvaluationSchema
        }
    )

    return {
        'depth_of_analysis_feedback': response.parsed.feedback,
        'individual_scores': [response.parsed.score]
    }

In [22]:
def language_used_evaluation(State: UPSCState) -> UPSCState:

    essay_generated = State['essay_generated']

    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10\n{essay_generated}",
        config = {
            'response_mime_type': 'application/json',
            'response_schema': EvaluationSchema
        }
    )

    return {
        'language_used_feedback': response.parsed.feedback,
        'individual_scores': [response.parsed.score]
    }

In [23]:
def generate_summary(State: UPSCState) -> UPSCState:

    essay_topic = State['essay_topic']

    clarity_of_thought_feedback = State['clarity_of_thought_feedback']

    depth_of_analysis_feedback = State['depth_of_analysis_feedback']

    language_used_feedback = State['language_used_feedback']

    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = f"Give me the combined feedback of the essay on the topic: {essay_topic} based on\nclarity_of_thought_feedback_text: {clarity_of_thought_feedback}\n\ndepth_of_analysis_feedback_text: {depth_of_analysis_feedback}\n\nlanguage_used_feedback_text: {language_used_feedback}\n"
    )

    return {
        'overall_feedback': response.text,
        'avg_score': sum(State['individual_scores']) / len(State['individual_scores'])
    }

In [24]:
graph = StateGraph(UPSCState)

graph.add_node('essay_generated', generate_essay)
graph.add_node('clarity_of_thought_evaluation', clarity_of_thought_evaluation)
graph.add_node('depth_of_analysis_evaluation', depth_of_analysis_evaluation)
graph.add_node('language_used_evaluation', language_used_evaluation)
graph.add_node('generate_summary', generate_summary)

graph.add_edge(START, 'essay_generated')
graph.add_edge('essay_generated', 'clarity_of_thought_evaluation')
graph.add_edge('essay_generated', 'depth_of_analysis_evaluation')
graph.add_edge('essay_generated', 'language_used_evaluation')
graph.add_edge('clarity_of_thought_evaluation', 'generate_summary')
graph.add_edge('depth_of_analysis_evaluation', 'generate_summary')
graph.add_edge('language_used_evaluation', 'generate_summary')
graph.add_edge('generate_summary', END)

workflow = graph.compile()

In [25]:
initial_state = {
    'essay_topic': "The MAJORANA Chip from Google"
}

final_state = workflow.invoke(initial_state)

UPSCState['avg_score']

__main__.UPSCState['avg_score']